<a href="https://colab.research.google.com/github/ghawkes333/KneeMRIClassifier/blob/main/explore_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GenAI: I used Claude and HuggingFaceAI to learn how to download a set of npz files from HuggingFaceHub. I used Gemini to draft code to load the model and run inference on MedGamma

Reference for fine tuning MedGamma: https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb

In [1]:
!pip install bitsandbytes
!pip install accelerate

In [2]:
from datasets import load_dataset
from huggingface_hub import login
from huggingface_hub import hf_hub_download
from datasets import Dataset, concatenate_datasets
from transformers import pipeline
import numpy as np
from huggingface_hub import snapshot_download
import glob

In [3]:
login()

In [4]:
# sing_coil = load_dataset("AUMLProject/fastmri-knee-singlecoil-rss")
def download_one_example():
  filepath1 = hf_hub_download(
      repo_id="AUMLProject/fastmri-knee-singlecoil-rss",
      filename="data/file1000001.npz",
      repo_type="dataset"
  )
  npz1 = np.load(filepath1)

  return npz1

In [5]:
example = download_one_example()
print(example)

NpzFile '/root/.cache/huggingface/hub/datasets--AUMLProject--fastmri-knee-singlecoil-rss/snapshots/be54b608d0ea125707a5bd69d4879fb27677c3a4/data/file1000001.npz' with keys: images, filename, num_slices, height, width...


In [6]:

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import bitsandbytes as bnb
import accelerate
from transformers import pipeline

In [7]:


# Load MedGemma (using 4-bit quantization to fit in Colab T4 RAM)
model_id = "google/medgemma-1.5-4b-it"
# processor = AutoProcessor.from_pretrained(model_id)



# model = AutoModelForImageTextToText.from_pretrained(
#     model_id,processor

#     # torch_dtype=torch.bfloat16,
#     device_map="auto",
#     offload_folder="offload",
#     offload_state_dict=True,
#     # load_in_4bit=True
# )

In [8]:


def predict_acl_draft(sample):
    # Extract the image (assuming 'images' is a path or a numpy array in the npz)
    # Most npz datasets store a stack; we'll take the middle slice
    mri_stack = np.load(sample['images'])['data'] if isinstance(sample['images'], str) else sample['images']
    mid_slice = mri_stack[int(sample['num_slices']) // 2]

    # Normalize and convert to PIL
    rescaled = (255.0 / mid_slice.max() * (mid_slice - mid_slice.min())).astype(np.uint8)
    image = Image.fromarray(rescaled).convert("RGB")
    image = image.resize((224, 224))

    # Prepare prompt
    prompt = "User: <image>\nDoes this knee MRI show an ACL (Anterior Cruciate Ligament) injury? Answer 'Yes' or 'No' and provide a brief reason.\nAssistant:"

    inputs = processor(text=prompt, images=image, return_tensors="pt", image_seq_length=1).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)

    prediction_text = processor.decode(output[0], skip_special_tokens=True)
    return prediction_text

In [9]:


def predict_acl(sample, pipe):
    # Extract the image (assuming 'images' is a path or a numpy array in the npz)
    # Most npz datasets store a stack; we'll take the middle slice
    mri_stack = np.load(sample['images'])['data'] if isinstance(sample['images'], str) else sample['images']
    mid_slice = mri_stack[int(sample['num_slices']) // 2]

    # Normalize and convert to PIL
    rescaled = (255.0 / mid_slice.max() * (mid_slice - mid_slice.min())).astype(np.uint8)
    image = Image.fromarray(rescaled).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Does this knee MRI show an ACL (Anterior Cruciate Ligament) injury? Answer 'Yes' or 'No' and provide a brief reason."},
                {"type": "image", "image": image}
            ]
        }
    ]

    # Prepare prompt
    # prompt = "User: <image>\nDoes this knee MRI show an ACL (Anterior Cruciate Ligament) injury? Answer 'Yes' or 'No' and provide a brief reason.\nAssistant:"

    # inputs = processor(text=prompt, images=image, return_tensors="pt", image_seq_length=1).to(model.device)
    with torch.no_grad():
      output = pipe(text=messages, max_new_tokens=50, do_sample=False, use_cache=False)
      response = output[0]["generated_text"][-1]["content"]

    return response

    # with torch.no_grad():
    #     output = model.generate(**inputs, max_new_tokens=50)

    # prediction_text = processor.decode(output[0], skip_special_tokens=True)
    # return prediction_text


In [10]:
# Run on the first file
# sample_file = ds[0]
from transformers import BitsAndBytesConfig
sample_file = example


model_kwargs = dict(
    # dtype=torch.bfloat16,
    device_map="auto",
    # offload_buffers=True
    quantization_config=BitsAndBytesConfig(load_in_4bit=True) # Loads the weights in 4bit pieces, allowing a larger model to fit. Speeds up inference but much slower to load initially
)

pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)
print(f"Predicting for: {sample_file['filename']}")
print(predict_acl(sample_file, pipe))

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Predicting for: file1000001


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No.

**Reason:** The image provided is a knee MRI, but it does not show clear anatomical structures like the ACL. The image is too dark and lacks the necessary contrast and detail to identify the ACL or other specific knee structures. An MRI


In [11]:
# TODO: Check if this downloads fully and concats the datasets correctly
def download_full_singlecoil_dataset():
  local_dir = snapshot_download(
      repo_id="AUMLProject/fastmri-knee-singlecoil-rss",
      repo_type="dataset",
      allow_patterns="*.npz",

  )

  datasets = []
  for filepath in sorted(glob.glob(f"{local_dir}/data/*.npz")):
      npz = np.load(filepath, allow_pickle=True)
      datasets.append(Dataset.from_dict({k: npz[k] for k in npz.files}))

  dataset = concatenate_datasets(datasets)

  return dataset

  print(local_dir)


In [12]:


pipe = pipeline("image-text-to-text", model="google/medgemma-1.5-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'input_text': [{'role': 'user',
    'content': [{'type': 'image',
      'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
     {'type': 'text', 'text': 'What animal is on the candy?'}]}],
  'generated_text': [{'role': 'user',
    'content': [{'type': 'image',
      'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
     {'type': 'text', 'text': 'What animal is on the candy?'}]},
   {'role': 'assistant',
    'content': 'The animal on the candy is a **bird**.\n'}]}]

In [13]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")
model = AutoModelForImageTextToText.from_pretrained("google/medgemma-1.5-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


The animal on the candy is a **bird**.
<end_of_turn>
